# Supplementary Demonstration: CODEBRIM 5-class (§5.7)

**Simulated BIM context.** Element-type assignment is derived from
the defect label itself, creating potential circularity. Results are
therefore supplementary evidence only; all statistical claims in the
paper are based on SDNET2018.

Multi-label → single-label conversion uses a dominant-defect priority
rule: exposed bars > spallation > corrosion > crack > efflorescence.

## Setup

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

with open('../data/experiment_A_results.json', 'r') as f:
    codebrim = json.load(f)

with open('../data/experiment_B_results.json', 'r') as f:
    pairwise = json.load(f)

with open('../data/experiment_C_results.json', 'r') as f:
    sensitivity = json.load(f)

print(f"CODEBRIM test images: {codebrim['n_test_images']}")
print(f"Classes: {list(codebrim['summary']['baseline']['per_class'].keys())}")

## BIM Prior Construction (Simulated Element Types)

In [ ]:
print('Simulated BIM priors (7 element types):')
print('=' * 60)
for elem, priors in codebrim['priors_used'].items():
    print(f"\n{elem}:")
    for cls, p in priors.items():
        bar = '█' * int(p * 20)
        print(f"  {cls:16s} {p:.2f} {bar}")

## Method Comparison (Table 5)

In [ ]:
summary = codebrim['summary']
print('CODEBRIM Method Comparison:')
print('=' * 55)
print(f"{'Method':12s} {'Accuracy':>10s} {'Macro F1':>10s}")
print('-' * 55)
for method in ['baseline', 'bayesian', 'oracle']:
    d = summary[method]
    print(f"{method:12s} {d['accuracy']:>10.4f} {d['macro_f1']:>10.4f}")

print(f"\nImprovement (Bayesian vs Baseline):")
print(f"  ΔAccuracy: +{(summary['bayesian']['accuracy']-summary['baseline']['accuracy'])*100:.2f}%")
print(f"  ΔMacro F1: +{(summary['bayesian']['macro_f1']-summary['baseline']['macro_f1'])*100:.2f}%")

## Pairwise Confusion Analysis

In [ ]:
print('Pairwise confusion reduction:')
print('=' * 75)
print(f"{'Actual':>16s} → {'Predicted':>16s}  {'Base':>5s} {'BPC':>5s} {'Δ':>4s} {'PCRR%':>6s}")
print('-' * 75)
for pair in pairwise['pair_results']:
    print(f"{pair['actual']:>16s} → {pair['predicted']:>16s}  "
          f"{pair['base_count']:>5d} {pair['bpc_count']:>5d} {pair['delta']:>+4d} "
          f"{pair['pcrr']:>+6.1f}")
print(f"\nTotal errors: {pairwise['total_errors_baseline']} → {pairwise['total_errors_bpc']} "
      f"({pairwise['overall_error_reduction_pct']:.1f}% reduction)")

## Sensitivity Analysis (CODEBRIM)

In [ ]:
alphas = sorted([float(a) for a in sensitivity.keys()])
acc = [sensitivity[str(a)]['accuracy'] for a in alphas]
f1 = [sensitivity[str(a)]['macro_f1'] for a in alphas]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(alphas, acc, 'o-', color='#2196F3', label='Accuracy', linewidth=2)
ax.plot(alphas, f1, 's-', color='#FF9800', label='Macro F1', linewidth=2)
ax.set_xlabel('α (prior degradation)')
ax.set_ylabel('Score')
ax.set_title('CODEBRIM: Prior sensitivity')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/nb07_codebrim_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

## Limitations

- **Circularity risk:** element-type is simulated from defect labels.
- **Multi-label simplification:** priority rule may inflate baseline errors.
- **No statistical significance claims** are made for CODEBRIM results.
- All inferential statistics (McNemar, bootstrap CIs) are based exclusively on SDNET2018.